# Module 05 - Embeddings and Positions

Use this notebook as the working space for the Module 05 exercises.

1. Read the lesson page (`docs/modules/05-embeddings.md`).
2. Open this notebook with `./notebook.sh 05`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from g2c.artifacts import load_corpus_text, load_tokenizer_artifact
from g2c.embeddings import (
    LearnedPositionalEmbedding,
    RotaryEmbedding,
    SinusoidalPositionalEmbedding,
    SkipGramEmbeddingModel,
    TokenEmbedding,
    analogy,
    load_glove_subset,
    make_skipgram_pairs,
    nearest_by_cosine,
    normalized,
    train_skipgram,
)
from g2c.notebook_extras.embeddings import (
    frequent_learned_token_ids,
    learned_token_vectors,
    plot_glove_slice,
    plot_learned_token_embeddings_2d,
    token_key,
)
from g2c.tokenizer import BPETokenizer

torch.manual_seed(0)


## Before the Notebook

Use the tests to implement the library pieces first. The notebook assumes `TokenEmbedding`, additive positional embeddings, and `RotaryEmbedding` become available as you progress through `tests/test_embeddings.py`.

In [ ]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_embeddings.py -x"
"Question: Which embeddings test is the next one failing, and what implementation does it point at?"
"Answer: "

In [ ]:
import subprocess
import sys
import g2c

repo_root = Path(g2c.__file__).resolve().parents[1]
test_path = repo_root / "tests" / "test_embeddings.py"
assert test_path.exists(), f"Could not find {test_path}"

result = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_path), "-q"],
    cwd=repo_root,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

assert result.returncode == 0, "Module 05 embedding tests are not passing yet."
print("Module 05 embedding tests passed.")

## Exercise 1 - Token and Learned Position Lookups

The core embedding operation is integer indexing into a learned table. Predict the shapes first, then use the checks below after the forwards are implemented.

In [ ]:
"Question: If ids has shape (B, T) and the embedding table has shape (V, C), what shape should weight[ids] have?"
"Answer: "

"Question: Why is the token embedding forward pass just table indexing, not matrix multiplication?"
"Answer: "

"Question: LearnedPositionalEmbedding returns shape (T, C). Why can it be added to token vectors shaped (B, T, C)?"
"Answer: "

In [ ]:
token_emb = TokenEmbedding(vocab_size=12, embedding_dim=4)
ids = torch.tensor([[3, 1, 7], [0, 5, 5]])
learned_pos = LearnedPositionalEmbedding(max_seq_len=16, embedding_dim=4)

token_vectors = token_emb(ids)
pos_vectors = learned_pos(seq_len=ids.shape[1])
combined = token_vectors + pos_vectors

print("ids:", tuple(ids.shape))
print("token vectors:", tuple(token_vectors.shape))
print("position vectors:", tuple(pos_vectors.shape))
print("combined:", tuple(combined.shape))
assert token_vectors.shape == (2, 3, 4)
assert pos_vectors.shape == (3, 4)
assert combined.shape == (2, 3, 4)
assert torch.allclose(token_vectors[0, 0], token_emb.weight[3])

## Exercise 2 - Sinusoidal Positional Encoding

Build the fixed sine/cosine table in `SinusoidalPositionalEmbedding.__init__`, then inspect the values and the heatmap.

In [ ]:
"Question: At position 0, what should every sine slot contain? What should every cosine slot contain?"
"Answer: "

"Question: Why does this implementation require an even embedding_dim?"
"Answer: "

"Question: Why should the sinusoidal table have no learnable parameters?"
"Answer: "

In [ ]:
sinusoidal = SinusoidalPositionalEmbedding(max_seq_len=64, embedding_dim=32)
prefix = sinusoidal(seq_len=8)
print("full table:", tuple(sinusoidal.weight.shape))
print("prefix:", tuple(prefix.shape))
print("position 0:", sinusoidal.weight[0, :8])
assert torch.allclose(sinusoidal.weight[0, 0::2], torch.zeros(16), atol=1e-6)
assert torch.allclose(sinusoidal.weight[0, 1::2], torch.ones(16), atol=1e-6)

plt.figure(figsize=(9, 4))
plt.imshow(sinusoidal.weight.T, aspect="auto", interpolation="nearest")
plt.xlabel("position")
plt.ylabel("dimension")
plt.title("Sinusoidal positional encoding")
plt.colorbar()
plt.show()

## Exercise 3 - RoPE Table Construction

`RotaryEmbedding.__init__` precomputes cosine and sine tables. The split-halves convention is what makes those tables align with `_rotate_half`.

In [ ]:
"Question: In the split-halves RoPE convention, which dimensions are paired together when embedding_dim is 8?"
"Answer: "

"Question: Why do the cos and sin tables both have shape (max_seq_len, embedding_dim), not (max_seq_len, embedding_dim // 2)?"
"Answer: "

"Question: What should cos[0] and sin[0] contain, and why?"
"Answer: "

In [ ]:
rope = RotaryEmbedding(max_seq_len=32, embedding_dim=8)
print("cos:", tuple(rope.cos.shape))
print("sin:", tuple(rope.sin.shape))
print("cos[0]:", rope.cos[0])
print("sin[0]:", rope.sin[0])
assert torch.allclose(rope.cos[0], torch.ones(8), atol=1e-6)
assert torch.allclose(rope.sin[0], torch.zeros(8), atol=1e-6)

## Exercise 4 - Apply RoPE

Once the tables exist, the forward pass is a rotation: `x * cos + rotate_half(x) * sin`. The important check is that dot products depend on relative position, not absolute position.

In [ ]:
"Question: Why does a rotation preserve the L2 norm of each vector?"
"Answer: "

"Question: Why is position 0 the identity rotation?"
"Answer: "

"Question: In attention, why is a relative-position dot-product property more useful than encoding only absolute positions?"
"Answer: "

In [ ]:
def rotated_dot(rope: RotaryEmbedding, q: torch.Tensor, k: torch.Tensor, m: int, n: int) -> float:
    """Return dot(R(q, m), R(k, n)) for one absolute position pair."""
    seq_len = max(m, n) + 1
    seq_q = torch.zeros(1, seq_len, q.shape[-1])
    seq_k = torch.zeros(1, seq_len, k.shape[-1])
    seq_q[0, m] = q
    seq_k[0, n] = k
    return (rope(seq_q)[0, m] * rope(seq_k)[0, n]).sum().item()


torch.manual_seed(0)
rope = RotaryEmbedding(max_seq_len=20, embedding_dim=16)
q = torch.randn(16)
k = torch.randn(16)
position_pairs = [(0, 2), (3, 5), (7, 9)]
same_offset_scores = [rotated_dot(rope, q, k, m, n) for m, n in position_pairs]

print("m  n  n-m  RoPE dot(q_m, k_n)")
print("-" * 33)
for (m, n), score in zip(position_pairs, same_offset_scores):
    print(f"{m:1d}  {n:1d}   {n - m:1d}   {score: .6f}")

score_spread = max(same_offset_scores) - min(same_offset_scores)
print(f"\nscore spread across same-offset pairs: {score_spread:.8f}")
assert max(same_offset_scores) - min(same_offset_scores) < 1e-4

## Exercise 5 - Train Tiny Co-occurrence Embeddings

This is a small skip-gram style model: choose a center token and predict nearby context tokens. If the reusable `ShakespeareTokenizer` artifact exists, the exercise uses its full 4096-token vocabulary on the full TinyShakespeare corpus. Otherwise it falls back to a smaller in-notebook tokenizer and corpus slice. Do not expect GloVe-style semantic analogies yet; that contrast is the point of Exercise 6.


In [ ]:
SHAKESPEARE_TOKENIZER_ARTIFACT = "ShakespeareTokenizer"
FALLBACK_TINY_SHAKESPEARE_CHARS = 120_000
FALLBACK_EMBEDDING_TOKENIZER_VOCAB = 768

try:
    repo_root
except NameError:
    import g2c

    repo_root = Path(g2c.__file__).resolve().parents[1]

tiny_corpus = load_corpus_text("tinyshakespeare", repo_root=repo_root)
assert tiny_corpus is not None, "Run ./setup.sh to download data/datasets/tinyshakespeare.txt first."

try:
    shakespeare_tokenizer_artifact = load_tokenizer_artifact(
        SHAKESPEARE_TOKENIZER_ARTIFACT,
        repo_root=repo_root,
    )
except FileNotFoundError:
    shakespeare_tokenizer_artifact = None

if shakespeare_tokenizer_artifact is not None:
    tiny_tokenizer = shakespeare_tokenizer_artifact.tokenizer
    corpus_ids = tiny_tokenizer.encode_with_vocab_size(tiny_corpus, None)
    tokenizer_mode = "loaded ShakespeareTokenizer artifact (full vocab)"
else:
    tiny_corpus = tiny_corpus[:FALLBACK_TINY_SHAKESPEARE_CHARS]
    tiny_tokenizer = BPETokenizer()
    corpus_ids = tiny_tokenizer.train(
        tiny_corpus,
        vocab_size=FALLBACK_EMBEDDING_TOKENIZER_VOCAB,
    )
    tokenizer_mode = "trained fallback tokenizer in notebook"

vocab_size = len(tiny_tokenizer.vocab)

print("tokenizer:", tokenizer_mode)
print("corpus characters:", len(tiny_corpus))
print("corpus tokens:", len(corpus_ids))
print("vocab size:", vocab_size)
print("first 40 ids:", corpus_ids[:40])
print("text preview:")
print(tiny_corpus[:240])


In [ ]:
"Question: In a skip-gram pair, which token is the input and which token is the target?"
"Answer: "

"Question: Why does predicting nearby tokens push co-occurring tokens toward useful embedding geometry?"
"Answer: "

"Question: What structure should TinyShakespeare reveal more clearly than the old hand-written corpus?"
"Answer: "

"Question: Why should you still expect TinyShakespeare embeddings to be weaker than pretrained word vectors?"
"Answer: "


In [ ]:
# make_skipgram_pairs lives in g2c/embeddings/skipgram.py.
# Implement it there so the center/context construction is tested.
make_skipgram_pairs


In [ ]:
# SkipGramEmbeddingModel and train_skipgram live in g2c/embeddings/skipgram.py.
SkipGramEmbeddingModel, train_skipgram


In [ ]:
centers, contexts = make_skipgram_pairs(corpus_ids, window=2)
skipgram = SkipGramEmbeddingModel(vocab_size=vocab_size, embedding_dim=48)
skipgram_losses = train_skipgram(
    skipgram,
    centers,
    contexts,
    steps=2000,
    batch_size=512,
    lr=0.2,
    generator=torch.Generator().manual_seed(5),
)
print("skip-gram pairs:", centers.numel())
print("first loss:", skipgram_losses[0])
print("final loss:", skipgram_losses[-1])
assert skipgram_losses[-1] < skipgram_losses[0]

plt.figure(figsize=(7, 4))
plt.plot(skipgram_losses)
plt.xlabel("step")
plt.ylabel("cross-entropy")
plt.title("TinyShakespeare skip-gram training loss")
plt.show()


In [ ]:
# PCA projection and readable-token helpers live in g2c/notebook_extras/embeddings.py.
# They are display glue, not part of the skip-gram implementation.
learned_token_vectors, frequent_learned_token_ids, token_key


In [ ]:
token_vectors = learned_token_vectors(skipgram, tiny_tokenizer, corpus_ids)
probe_ids = frequent_learned_token_ids(corpus_ids, tiny_tokenizer, top_n=8, min_chars=3)

print("Nearest learned tokens by cosine similarity")
print("-" * 88)
for token_id in probe_ids:
    label = token_key(tiny_tokenizer, token_id)
    neighbors = nearest_by_cosine(
        skipgram.embedding.weight[token_id].detach(),
        token_vectors,
        exclude={label},
        top_k=5,
    )
    neighbor_text = ", ".join(f"{word} ({score:.2f})" for word, score in neighbors)
    print(f"{label:<22} -> {neighbor_text}")


In [ ]:
plot_learned_token_embeddings_2d(skipgram, tiny_tokenizer, corpus_ids)


## Exercise 6 - Pretrained Vector Analogies

`./datasets.sh glove` prepares `data/embeddings/glove.6B.50d.txt` for this exercise. The helper loads only the requested words, so it does not need to keep the full file in memory. If the file is missing, the notebook skips the pretrained section.

In [ ]:
"Question: What vector expression should approximate queen in the classic analogy?"
"Answer: "

"Question: Why is cosine similarity usually better than raw dot product for nearest-neighbor lookup in pretrained embeddings?"
"Answer: "

"Question: If the pretrained analogy works but your tiny model does not, what does that say about corpus scale and training signal?"
"Answer: "

"Question: In the 2D GloVe plot, which clusters or analogy offsets are visible? Which ones are hard to see after projection?"
"Answer: "

In [ ]:
# load_glove_subset, normalized, nearest_by_cosine, and analogy live in
# g2c/embeddings/similarity.py. This cosine-neighbor idea comes back in
# Module 17 when retrieval ranks chunks by vector similarity.
load_glove_subset, normalized, nearest_by_cosine, analogy


In [ ]:
try:
    repo_root
except NameError:
    import g2c

    repo_root = Path(g2c.__file__).resolve().parents[1]

glove_path = repo_root / "data" / "embeddings" / "glove.6B.50d.txt"
analogy_words = {
    "king", "queen", "man", "woman", "prince", "princess",
    "crown", "throne", "palace", "royal",
    "paris", "france", "italy", "rome", "madrid", "spain",
    "berlin", "germany", "london", "england",
    "cat", "dog", "kitten", "puppy", "animal", "pet",
}

if glove_path.exists():
    pretrained = load_glove_subset(glove_path, analogy_words)
    print("king - man + woman:", analogy("king", "man", "woman", pretrained))
    print("paris - france + italy:", analogy("paris", "france", "italy", pretrained))
else:
    print(f"Skipping pretrained analogies; add a local GloVe file at {glove_path}.")

In [ ]:
# GloVe projection and plotting glue lives in g2c/notebook_extras/embeddings.py.
if "pretrained" in globals():
    plot_glove_slice(pretrained)
else:
    print("Run the GloVe loading cell above first.")


## Exercise 7 - Compare Positional Schemes

Plot learned, sinusoidal, and RoPE tables side-by-side. The learned table is intentionally untrained here, so it should look random. Sinusoidal and RoPE show structure immediately because their tables come from fixed formulas.


In [ ]:
"Question: Why does the learned positional table look noisy before training?"
"Answer: "

"Question: Which positional scheme has learnable parameters?"
"Answer: "

"Question: What visual pattern do sinusoidal and RoPE tables share?"
"Answer: "

"Question: Why is RoPE applied to queries and keys inside attention rather than simply added to token embeddings here?"
"Answer: "


In [ ]:
max_seq_len = 64
embedding_dim = 32
learned = LearnedPositionalEmbedding(max_seq_len=max_seq_len, embedding_dim=embedding_dim)
sinusoidal = SinusoidalPositionalEmbedding(max_seq_len=max_seq_len, embedding_dim=embedding_dim)
rope = RotaryEmbedding(max_seq_len=max_seq_len, embedding_dim=embedding_dim)

tables = [
    ("learned (untrained)", learned.weight.detach()),
    ("sinusoidal", sinusoidal.weight.detach()),
    ("RoPE cos", rope.cos.detach()),
]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, (title, table) in zip(axes, tables):
    im = ax.imshow(table.T, aspect="auto", interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("position")
    ax.set_ylabel("dimension")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()
